# Lab 03: Developing a Tiny Transformer Language Model

This practical trains a small character-level Transformer from scratch on a shuffled subset of the Hugging Face [roneneldan/TinyStories](https://huggingface.co/datasets/roneneldan/TinyStories) corpus. It explores the data, builds next-character examples, implements causal self-attention and Transformer blocks, monitors training and validation loss, evaluates perplexity, compares settings, generates samples, and saves a reloadable checkpoint.

The earlier 300-story/200-update run produced incoherent samples. This revised run increases the training subset and optimization budget substantially. Run all cells in Google Colab with a GPU runtime; generated samples must still be reviewed because a small model trained from scratch is not guaranteed to produce consistently coherent stories. Dataset attribution: TinyStories, provided under [CDLA-Sharing-1.0](https://cdla.dev/sharing-1-0/).

In [ ]:
%pip -q install --upgrade "huggingface_hub>=1.32.0,<3.0" "gradio>=6.31.0,<7.0"


In [ ]:
from pathlib import Path
from itertools import islice
import json, math, os, random, tempfile
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from IPython.display import display

SEED = 42
SAMPLE_SIZE = 20000
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
torch.set_num_threads(min(2, os.cpu_count() or 1))
print(f"Device: {DEVICE}; shuffled TinyStories records requested: {SAMPLE_SIZE:,}")
if DEVICE.type == "cpu":
    print("WARNING: CPU training for this configuration may be very slow. In Colab select Runtime > Change runtime type > T4 GPU.")

## 1. Load and explore the Hugging Face dataset

In [ ]:
sample_path = os.environ.get("TINYSTORIES_SAMPLE_PATH")
if sample_path and Path(sample_path).exists():
    stories = json.loads(Path(sample_path).read_text(encoding="utf-8"))
else:
    stream = load_dataset("roneneldan/TinyStories", split="train", streaming=True, cache_dir=str(Path(tempfile.gettempdir()) / "tinystories_hf_cache"))
    stream = stream.shuffle(seed=SEED, buffer_size=min(SAMPLE_SIZE, 10_000))
    stories = list(islice(stream, SAMPLE_SIZE))
if not stories or "text" not in stories[0]:
    raise ValueError("Expected TinyStories records with a text field.")
lengths = [len(item["text"]) for item in stories]
print(f"Loaded {len(stories):,} shuffled stories. Character length summary:")
display(pd.Series(lengths, name="characters").describe(percentiles=[.25, .5, .75]).to_frame().round(1))
for i, item in enumerate(stories[:3], 1):
    print("\n--- Story {} ({} characters) ---\n{}\n".format(i, len(item["text"]), item["text"][:1200]))

## 2–3. Character tokenization and next-token training sequences

In [ ]:
indices = list(range(len(stories)))
random.Random(SEED).shuffle(indices)
cut = max(1, int(.9 * len(indices)))
train_stories = [stories[i]["text"].strip() for i in indices[:cut] if stories[i]["text"].strip()]
val_stories = [stories[i]["text"].strip() for i in indices[cut:] if stories[i]["text"].strip()]
train_text = chr(10).join(train_stories)
val_text = chr(10).join(val_stories)
UNK = "<unk>"
itos = [UNK] + sorted(set(train_text))
stoi = {char: i for i, char in enumerate(itos)}
encode = lambda text: [stoi.get(char, 0) for char in text]
decode = lambda ids: "".join(itos[i] for i in ids if i < len(itos) and itos[i] != UNK)
train_ids = torch.tensor(encode(train_text), dtype=torch.long)
val_ids = torch.tensor(encode(val_text), dtype=torch.long)
BLOCK_SIZE, BATCH_SIZE = 128, 32
print(f"Train/validation stories: {len(train_stories):,}/{len(val_stories):,}")
print(f"Train/validation characters: {len(train_ids):,}/{len(val_ids):,}; vocabulary: {len(itos)}")
def get_batch(data, batch_size=BATCH_SIZE, block_size=BLOCK_SIZE):
    if len(data) <= block_size + 1:
        raise ValueError("Not enough text to create a batch.")
    starts = torch.randint(0, len(data) - block_size - 1, (batch_size,))
    x = torch.stack([data[s:s + block_size] for s in starts])
    y = torch.stack([data[s + 1:s + block_size + 1] for s in starts])
    return x.to(DEVICE), y.to(DEVICE)
x_example, y_example = get_batch(train_ids, batch_size=1, block_size=30)
print("Input/target shapes:", tuple(x_example.shape), tuple(y_example.shape))
print("Input:", repr(decode(x_example[0].tolist())))
print("Next-character target:", repr(decode(y_example[0].tolist())))

## 4–5. Build and initialize the Transformer

In [ ]:
class CausalSelfAttention(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.attn = nn.MultiheadAttention(n_embd, n_head, dropout=dropout, batch_first=True)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        steps = x.size(1)
        mask = torch.triu(torch.ones(steps, steps, device=x.device, dtype=torch.bool), diagonal=1)
        attended, _ = self.attn(x, x, x, attn_mask=mask, need_weights=False)
        return self.dropout(attended)

class TransformerBlock(nn.Module):
    def __init__(self, n_embd, n_head, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(n_embd)
        self.attention = CausalSelfAttention(n_embd, n_head, dropout)
        self.ln2 = nn.LayerNorm(n_embd)
        self.ff = nn.Sequential(
            nn.Linear(n_embd, 4 * n_embd), nn.GELU(),
            nn.Linear(4 * n_embd, n_embd), nn.Dropout(dropout)
        )

    def forward(self, x):
        x = x + self.attention(self.ln1(x))
        return x + self.ff(self.ln2(x))

class TinyTransformer(nn.Module):
    def __init__(self, vocab_size, block_size, n_embd=64, n_head=4, n_layer=2, dropout=.1):
        super().__init__()
        if n_embd % n_head:
            raise ValueError("n_embd must be divisible by n_head")
        self.block_size = block_size
        self.token_embedding = nn.Embedding(vocab_size, n_embd)
        self.position_embedding = nn.Embedding(block_size, n_embd)
        self.embedding_dropout = nn.Dropout(dropout)
        self.blocks = nn.Sequential(*[
            TransformerBlock(n_embd, n_head, dropout) for _ in range(n_layer)
        ])
        self.final_norm = nn.LayerNorm(n_embd)
        self.head = nn.Linear(n_embd, vocab_size, bias=False)
        self.head.weight = self.token_embedding.weight
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0., std=.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.LayerNorm):
            nn.init.ones_(module.weight)
            nn.init.zeros_(module.bias)

    def forward(self, ids, targets=None):
        _, steps = ids.shape
        positions = torch.arange(steps, device=ids.device)
        x = self.embedding_dropout(
            self.token_embedding(ids) + self.position_embedding(positions)
        )
        logits = self.head(self.final_norm(self.blocks(x)))
        loss = None if targets is None else F.cross_entropy(
            logits.reshape(-1, logits.size(-1)), targets.reshape(-1)
        )
        return logits, loss

    @torch.no_grad()
    def generate(self, ids, max_new_tokens=400, temperature=.8, top_k=20):
        self.eval()
        for _ in range(max_new_tokens):
            context = ids[:, -self.block_size:]
            logits, _ = self(context)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            if top_k is not None:
                threshold = torch.topk(logits, min(top_k, logits.size(-1))).values[:, -1, None]
                logits = logits.masked_fill(logits < threshold, float("-inf"))
            next_id = torch.multinomial(torch.softmax(logits, dim=-1), 1)
            ids = torch.cat((ids, next_id), dim=1)
        return ids

print("Architecture ready: causal attention, token/position embeddings, MLP, residuals, LayerNorm, dropout.")

## 6–7. Train and monitor training/validation loss

In [ ]:
@torch.no_grad()
def estimate_loss(model, batches=20):
    model.eval()
    values = {}
    for name, data in [("train", train_ids), ("validation", val_ids)]:
        losses = []
        for _ in range(batches):
            xb, yb = get_batch(data)
            _, loss = model(xb, yb)
            losses.append(loss.item())
        values[name] = float(np.mean(losses))
    model.train()
    return values

def train_model(config, total_steps, run_name="baseline", eval_every=250):
    torch.manual_seed(SEED)
    model = TinyTransformer(len(itos), BLOCK_SIZE, **config).to(DEVICE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4, weight_decay=.01)
    history, best_val, best_state = [], float("inf"), None
    for step in range(1, total_steps + 1):
        model.train()
        xb, yb = get_batch(train_ids)
        _, loss = model(xb, yb)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step % eval_every == 0 or step == total_steps:
            measured = estimate_loss(model)
            row = {"run": run_name, "step": step, "train_loss": measured["train"], "validation_loss": measured["validation"]}
            history.append(row)
            print(f"{run_name} step {step:,}/{total_steps:,}: train={row['train_loss']:.3f}, val={row['validation_loss']:.3f}")
            if row["validation_loss"] < best_val:
                best_val = row["validation_loss"]
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, history

baseline_config = {"n_embd": 128, "n_head": 4, "n_layer": 4, "dropout": .1}
TRAIN_STEPS = 5000
model, history = train_model(baseline_config, total_steps=TRAIN_STEPS, run_name="baseline_4_layer")

## 8. Evaluate validation loss and perplexity; plot loss curves

In [ ]:
@torch.no_grad()
def full_validation_loss(model, batches=100):
    model.eval()
    losses = []
    for _ in range(batches):
        xb, yb = get_batch(val_ids)
        _, loss = model(xb, yb)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))
val_loss = full_validation_loss(model)
perplexity = math.exp(val_loss)
print(f"Best-checkpoint validation loss: {val_loss:.4f}")
print(f"Character perplexity: {perplexity:.2f}")
results_dir = Path.cwd().parent / "results"
results_dir.mkdir(parents=True, exist_ok=True)
history_df = pd.DataFrame(history)
history_df.to_csv(results_dir / "training_loss.csv", index=False)
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(history_df.step, history_df.train_loss, marker="o", label="Training loss")
ax.plot(history_df.step, history_df.validation_loss, marker="o", label="Validation loss")
ax.set(title="Tiny Transformer training progress", xlabel="Optimizer step", ylabel="Cross-entropy loss")
ax.grid(alpha=.25)
ax.legend()
fig.tight_layout()
fig.savefig(results_dir / "training_loss.svg", bbox_inches="tight")
fig.savefig(results_dir / "training_loss.png", dpi=160, bbox_inches="tight")
display(fig)
plt.close(fig)
display(history_df.round(3))

## 9. Generate a story from a prompt

In [ ]:
def generate_story(model, prompt="Once upon a time", max_new_tokens=400, temperature=.8, top_k=20):
    prompt_ids = torch.tensor([encode(prompt)], dtype=torch.long, device=DEVICE)
    generated = model.generate(prompt_ids, max_new_tokens=max_new_tokens, temperature=temperature, top_k=top_k)[0].tolist()
    return decode(generated)
baseline_story = generate_story(model)
print(baseline_story)

## 10. Change model settings and compare

In [ ]:
small_config = {"n_embd": 96, "n_head": 4, "n_layer": 2, "dropout": .1}
small_model, small_history = train_model(small_config, total_steps=1500, run_name="smaller_2_layer", eval_every=250)
small_loss = full_validation_loss(small_model)
comparison = pd.DataFrame([
    {"setting": f"4 layers, 4 heads, {TRAIN_STEPS} steps", "parameters": sum(p.numel() for p in model.parameters()), "validation_loss": val_loss, "perplexity": math.exp(val_loss)},
    {"setting": "2 layers, 4 heads, 1500 steps", "parameters": sum(p.numel() for p in small_model.parameters()), "validation_loss": small_loss, "perplexity": math.exp(small_loss)}
])
display(comparison.round(3))
comparison.to_csv(results_dir / "settings_comparison.csv", index=False)
pd.concat([history_df, pd.DataFrame(small_history)], ignore_index=True).to_csv(results_dir / "all_training_loss.csv", index=False)
small_story = generate_story(small_model, max_new_tokens=300)
print("Smaller model sample:")
print(small_story)

## 11. Save the trained model and reload it for generation

In [ ]:
model_config = {"vocab_size": len(itos), "block_size": BLOCK_SIZE, **baseline_config}
checkpoint_path = results_dir / "tiny_transformer.pt"
torch.save({"model_state_dict": model.state_dict(), "model_config": model_config, "itos": itos, "seed": SEED}, checkpoint_path)
config_report = {**model_config, "seed": SEED, "sample_size": len(stories), "train_stories": len(train_stories), "validation_stories": len(val_stories), "training_steps": TRAIN_STEPS, "tokenization": "character-level", "validation_loss": val_loss, "character_perplexity": perplexity}
(results_dir / "tiny_transformer_config.json").write_text(json.dumps(config_report, indent=2), encoding="utf-8")
checkpoint = torch.load(checkpoint_path, map_location=DEVICE, weights_only=True)
reloaded = TinyTransformer(**checkpoint["model_config"]).to(DEVICE)
reloaded.load_state_dict(checkpoint["model_state_dict"])
reloaded_story = generate_story(reloaded, max_new_tokens=250)
print("Saved {} ({} bytes)".format(checkpoint_path.name, checkpoint_path.stat().st_size))
print("Reloaded model sample:")
print(reloaded_story)
report_lines = [
    "# Generated TinyStories Samples", "",
    "Training data: {:,} shuffled TinyStories records; training steps: {:,}; validation loss: {:.4f}; character perplexity: {:.2f}.".format(len(stories), TRAIN_STEPS, val_loss, perplexity), "",
    "Please review these samples for coherence before treating them as successful outputs.", "",
    "## Baseline (4 layers, 4 heads)", "", baseline_story, "",
    "## Smaller comparison (2 layers, 4 heads)", "", small_story, "",
    "## Reloaded checkpoint sample", "", reloaded_story, ""
]
(results_dir / "generated_stories.md").write_text("\n".join(report_lines), encoding="utf-8")
print("Saved model weights, config, loss logs, settings comparison, generated samples, and curve in results/.")

## Performance mechanisms and observations

- **Causal attention mask:** blocks future characters from influencing next-character predictions.
- **Residual connections and pre-LayerNorm:** support gradient flow and training stability.
- **Dropout and AdamW weight decay:** regularize the model.
- **Gradient clipping:** limits unusually large updates.
- **More data and optimization:** the revised run samples 20,000 shuffled TinyStories records and performs 5,000 updates for the main model; a smaller comparison model runs 1,500 updates. Best validation checkpoints are restored.
- **Validation loss and perplexity:** monitor generalization and compare settings.

A small character-level Transformer trained from scratch can still make mistakes or produce inconsistent stories. Treat the generated samples as evidence to inspect, not as a guarantee of human-quality text. If loss is still falling at the end, increase TRAIN_STEPS and rerun; if validation loss rises while training loss falls, reduce the steps or add regularization.

## Generate stories from a chat prompt

Enter a story opening in the chat box, and the trained Tiny Transformer will continue it. This is a prompt-to-story demonstration, not a general question-answering assistant. Since the model is small and trained from scratch, generated text may be incoherent. The temporary Gradio share link is accessible to anyone who has it while the Colab runtime is active.

In [ ]:
import gradio as gr

def story_chat(message, history):
    prompt = (message or "").strip()
    if not prompt:
        return "Please enter a story opening, such as: Once upon a time there was a little fox."
    return generate_story(model, prompt=prompt, max_new_tokens=300, temperature=0.75, top_k=20)

story_app = gr.ChatInterface(
    fn=story_chat,
    title="TinyStories Story Generator",
    description="Enter a story opening and the Tiny Transformer will continue it. This small experimental model may produce errors or incoherent text.",
    examples=["Once upon a time there was a little fox.", "One sunny morning, Mia found a tiny boat."],
    type="messages",
)
story_app.launch(share=True, debug=False)